In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os

print(os.listdir(path))
csv_path = os.path.join(path,"Q1_data.csv")
df = pd.read_csv(csv_path)


In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df, "Delivery_Time")


In [ ]:
# Task 1: Write your code here:
df.drop(columns=['Order_ID'], inplace=True)

In [ ]:
# Task 2: Write your code here:
def check_missing_values(df):
    missing_values = df.isnull().sum()
    print("Missing Values per Column:")
    missing_data = missing_values[missing_values > 0]

    if not missing_data.empty:
        percentage = (missing_data / len(df)) * 100
        result = pd.concat([missing_data, percentage], axis=1, keys=['Count', 'Percentage'])
        print(result[result['Count'] > 0].round(2))
        print("\nHandle Missing Values as needed.")
    else:
        print("\nNo Missing Values Found.")
check_missing_values(df)

In [ ]:
# Task 2: Write your code here:
df['Weather'].fillna(df['Weather'].mode()[0], inplace=True)
df['Traffic_Level'].fillna(df['Traffic_Level'].mode()[0], inplace=True)
df['Time_of_Day'].fillna(df['Time_of_Day'].mode()[0], inplace=True)
df['Courier_Experience_yrs'].fillna(df['Courier_Experience_yrs'].mean(), inplace=True)
df['Delivery_Time'].fillna(df['Delivery_Time'].mean(), inplace=True)


In [ ]:
df.isnull().sum()

In [ ]:
# Task 3: Write your code here:
# Do we have duplicate samples?
def check_duplicates(df):
    duplicates = df.duplicated().sum()
    print(f"Number of Duplicate Samples: {duplicates}")
    if duplicates > 0:
        print("Dropping Duplicates...")
        df.drop_duplicates(inplace=True)
        print("Duplicates Dropped.")
    else:
        print("No Duplicate Samples Found.")


check_duplicates(df)

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import OneHotEncoder
# Do we have categorical columns?
categorical_cols = df.select_dtypes(include=["object"]).columns
print("Categorical Columns:", list(categorical_cols))




In [ ]:
pd.get_dummies(df, columns=['Weather','Traffic_Level','Time_of_Day','Vehicle_Type'], drop_first=True)

In [ ]:
df.head()

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import MinMaxScaler
numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET
scaler = MinMaxScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df

In [ ]:
# Task 6: Write your code here:
# 1. Is the target imbalanced?
def check_target_imbalance(df, target_column):
  print("Target Distribution:")

  df[target_column].hist()  # Yeah you can just do this :)
  plt.show()

check_target_imbalance(df, "Delivery_Time")


In [ ]:
# Task 1: Write your code here:
X = df.drop("Delivery_Time",axis=1)
y = df['Delivery_Time']

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(n_estimators=200)
# 5-Fold Cross-Validation, shuffled

scores_mae=[]
n_splits = 5  # K=5 Folds
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]



  # Train
  model.fit(X_train, y_train)

  # Predict
  y_pred = model.predict(X_test)

  # Calculate metrics

  scores_mae.append(mean_absolute_error(y_test, y_pred))


  # Store results
  print(f"{model} MAE Score: {np.mean(scores_mae)}")




In [ ]:
# Task 1: Write your code here:


importances = {}

importances['Random Forest'] = model.feature_importances_
# Create a 1x3 plot
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes = axes.flatten()
features = X.columns

for i, (model_name, imp) in enumerate(importances.items()):
  # Sort features by importance for a cleaner plot
  sorted_idx = np.argsort(imp)

  ax = axes[i]
  ax.barh(features[sorted_idx], imp[sorted_idx])
  ax.set_title(f"{model_name} Feature Importance")
  ax.set_xlabel("Importance Score")

plt.tight_layout()
plt.show()

In [ ]:
# Task Bonus: Write your code here:

In [ ]:
# Calculate the baseline predictions (mean of the target)
baseline_pred = np.full_like(y, y.mean())

# Evaluate the baseline
baseline_mae = mean_absolute_error(y, baseline_pred)

print(f"Baseline MAE (using mean target): {baseline_mae:.4f}")